# 03 — Cephalometric angular analysis

This notebook computes SNA, SNB, and signed ANB = SNA − SNB from source-coordinate reference annotations and predictions from the already-selected final checkpoint. It does not train, fine-tune, rank cases, change the cohort, or select a checkpoint.

Approved unrounded ANB-derived categories: Class III for ANB < 0°, Class I for 0° ≤ ANB ≤ 4°, and Class II for ANB > 4°. These are comparison categories, not complete clinical diagnoses or Angle classifications. Reference annotations are comparison references, not error-free anatomical truth.

The 60-image validation cohort informed model development/model selection and is not independent testing. Bland–Altman limits are descriptive, not clinical acceptability bounds. Angles are calculated in source coordinates, never resized-grid coordinates. Verify source geometry and square pixels or provide verified physical x/y spacing with units; square-pixel confirmation records dimensionless relative spacing, not mm/pixel. Supply physical spacing units explicitly when providing physical x/y spacing. Unknown geometry blocks execution. Inference determinism depends on the installed TensorFlow/device kernels and has not been verified on the target stack. The supplied best-MRE checkpoint is loaded strictly. The local checkout lacks the final checkpoint and matching data; attach them in Kaggle or configure paths below.

In [6]:
from pathlib import Path
import sys, hashlib, json, platform, csv, os
import numpy as np
HERE=Path.cwd().resolve()
FINAL_SETUP=Path("/kaggle/input/models/pouyansajadi78/model-oct-22-v2/tensorflow2/default/1")
if not (FINAL_SETUP/'src'/'clinical_analysis.py').is_file(): raise FileNotFoundError('Run from repository root or set FINAL_SETUP to Final Setup.')
sys.path=[str(FINAL_SETUP)]+[p for p in sys.path if Path(p or '.').resolve()!=FINAL_SETUP.parent]
import src as final_src
assert Path(final_src.__file__).resolve().parent==FINAL_SETUP/'src', final_src.__file__
from src import clinical_analysis as ca, clinical_plots as cp
from src.config import make_finetune_experiment_config, FINETUNE_TRAIN_IDS, CLINICAL_LANDMARK_INDICES
from src.reproducibility import set_global_seed
from src.data import read_points, read_rgb, preprocess
from src.model import build_model
CHECKPOINT=Path('/kaggle/input/models/pouyansajadi78/finetuned-model-oct22/tensorflow2/default/1/checkpoints/best_model_finetune_v3_finetune_n20_best_mre.weights.h5')
FT_ROOT=Path(os.getenv('CEPHALO_FT_ROOT','/kaggle/input/datasets/pouyansajadi78/fine-tuning-dataset-final'))
IMAGE_ROOT=FT_ROOT/'images_v2'/'images_v2'
ANNOTATION_ROOT=FT_ROOT/'annotations_v2'/'annotations_v2'
TRAIN_IDS=(25,76,80,71,46,24,41,72,31,16,42,15,22,69,33,38,5,50,9,47)
COHORT_IDS=(23,74,78,3,11,7,32,26,27,34,14,62,36,20,21,6,40,10,8,19,39,18,30,4,60,28,2,55,49,37,35,51,68,13,59,70,17,63,73,48,77,1,61,75,66,45,79,43,58,67,53,64,56,52,29,54,57,65,12,44)
# Confirm square-pixel geometry; relative spacing is dimensionless, not mm/pixel.
SQUARE_PIXELS_CONFIRMED=True
SPACING_X_PHYSICAL=None
SPACING_Y_PHYSICAL=None
PHYSICAL_SPACING_UNITS=None
COORDINATE_ORIGIN='top_left_pixel_center'
COORDINATE_ORIENTATION='x_right_y_down'
SEED,BOOTSTRAP_REPETITIONS=42,5000
OUTPUT_BASE=Path('/kaggle/working/clinical_angular_analysis')
RUN_ID='validation_best_mre_checkpoint'
print('Final Setup:',FINAL_SETUP,'| selected experiment: finetune_n20 | validation:',len(COHORT_IDS))

Final Setup: /kaggle/input/models/pouyansajadi78/model-oct-22-v2/tensorflow2/default/1 | selected experiment: finetune_n20 | validation: 60


## 1. Verify selected run, cohort, inputs, and pixel geometry
The cohort IDs and training split reproduce the executed notebook-2 run. The selected checkpoint filename matches the executed notebook-2 finetune_n20 best-MRE artifact. Notebook 2 selected its best-MRE artifact by validation MRE; it used a configured preprocessing resolution of 0.08 mm/pixel. This does not establish source-image x/y spacing or square-pixel geometry. Verify checkpoint provenance and acquisition geometry before analysis.

In [8]:
if not CHECKPOINT.is_file(): raise FileNotFoundError(f'Final selected checkpoint unavailable: {CHECKPOINT}')
if not IMAGE_ROOT.is_dir() or not ANNOTATION_ROOT.is_dir(): raise FileNotFoundError(f'Configure correct image/annotation roots: {IMAGE_ROOT}, {ANNOTATION_ROOT}')
split=make_finetune_experiment_config(TRAIN_IDS,COHORT_IDS,experiment_name='finetune_n20',trainable_scope='dec2_head_logits')
if len(split.train_image_ids)!=20 or len(split.val_image_ids)!=60 or len(set(TRAIN_IDS))!=20 or len(set(COHORT_IDS))!=60 or min(TRAIN_IDS+COHORT_IDS)<1 or max(TRAIN_IDS+COHORT_IDS)>80 or set(TRAIN_IDS)&set(COHORT_IDS): raise ValueError('Expected unique, in-range, disjoint recorded 20/60 run split.')
if SQUARE_PIXELS_CONFIRMED:
    SPACING=(1.,1.); SPACING_UNITS='dimensionless_relative'; PHYSICAL_SPACING=None
elif SPACING_X_PHYSICAL is not None and SPACING_Y_PHYSICAL is not None and PHYSICAL_SPACING_UNITS:
    SPACING=(float(SPACING_X_PHYSICAL),float(SPACING_Y_PHYSICAL)); SPACING_UNITS='physical'; PHYSICAL_SPACING=(SPACING_X_PHYSICAL,SPACING_Y_PHYSICAL,PHYSICAL_SPACING_UNITS)
else: raise ValueError('Verify square pixels or supply verified physical x/y spacing and units; do not assume geometry.')
if min(SPACING)<=0: raise ValueError('Axis spacings must be positive.')
if dict(CLINICAL_LANDMARK_INDICES)!={'Sella':0,'Nasion':1,'Subspinale':4,'Supramentale':5}: raise ValueError('Unexpected clinical landmark mapping.')
cfg=make_finetune_experiment_config(TRAIN_IDS,COHORT_IDS,experiment_name='finetune_n20',trainable_scope='dec2_head_logits',ft_root=str(FT_ROOT),image_resolution=0.08,backbone_weights=None)
assert cfg.decoder_dropout_rate==0.20 and cfg.encoder_mode=='stage1' and cfg.freeze_encoder_batch_norm
OUTPUT_DIR=OUTPUT_BASE/RUN_ID
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()): raise FileExistsError(f'Choose another RUN_ID; output folder already populated: {OUTPUT_DIR}')
print('Verified relative axis spacing:',SPACING,SPACING_UNITS,'| physical spacing:',PHYSICAL_SPACING,'| inference cfg:',cfg.stage_name,cfg.experiment_name)

Verified relative axis spacing: (1.0, 1.0) dimensionless_relative | physical spacing: None | inference cfg: stage2_finetune_v3 finetune_n20


## 2. Strict inference and source-coordinate export
Use the same model output contract as notebook 2 (`coords` normalized to [0,1]). Read original annotation coordinates directly, perform the configured unaugmented preprocessing, and restore predictions with each source image's actual width/height.

In [9]:
from src.evaluate import load_compatible_weights
set_global_seed(SEED)
model,encoder_names=build_model(cfg)
load_compatible_weights(model,str(CHECKPOINT),cfg)  # strict; no skipped weights
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
LANDMARK_NAMES=[cfg.anatomical_landmarks[str(i)] for i in range(cfg.num_landmarks)]
assert [LANDMARK_NAMES[i] for i in (0,1,4,5)]==['Sella','Nasion','Subspinale','Supramentale']
def find_unique(root,image_id,suffixes):
    candidates=[root/f'({image_id}){s}' for s in suffixes]+[root/f'{image_id}{s}' for s in suffixes]
    found=list(dict.fromkeys(p for p in candidates if p.is_file()))
    if len(found)!=1: raise FileNotFoundError(f'Expected one source file for ID {image_id}; found {found}')
    return found[0]
coordinate_rows=[]
for image_id in sorted(COHORT_IDS):
    image_path=find_unique(IMAGE_ROOT,image_id,('.tif','.tiff','.png','.jpg','.jpeg'))
    annotation_path=find_unique(ANNOTATION_ROOT,image_id,('.txt',))
    raw_image=read_rgb(image_path)
    height,width=raw_image.shape[:2]
    raw_truth=read_points(annotation_path,cfg)
    model_image,_=preprocess(raw_image,raw_truth,cfg,augmentation=None)
    outputs=model(np.expand_dims(model_image,axis=0),training=False)
    pred_norm=np.asarray(outputs['coords'])[0].astype(np.float64)
    if pred_norm.shape!=(cfg.num_landmarks,2) or not np.isfinite(pred_norm).all(): raise ValueError(f'{image_id}: invalid normalized prediction output')
    pred_raw=pred_norm*np.array([width,height],dtype=np.float64)
    for index,name in enumerate(LANDMARK_NAMES):
        coordinate_rows.append({'image_id':int(image_id),'landmark_name':name,'reference_x':float(raw_truth[index,0]),'reference_y':float(raw_truth[index,1]),'predicted_x':float(pred_raw[index,0]),'predicted_y':float(pred_raw[index,1]),'coordinate_system':'source_image_pixels_xy','source_width':width,'source_height':height,'x_spacing':SPACING[0],'y_spacing':SPACING[1],'spacing_units':SPACING_UNITS,'spacing_unit_name':PHYSICAL_SPACING_UNITS if SPACING_UNITS=='physical' else None,'origin':COORDINATE_ORIGIN,'orientation':COORDINATE_ORIENTATION})
    print(f'Inference complete: ID {image_id}, source {width}×{height}')
coordinate_export=OUTPUT_DIR/'source_coordinate_predictions_all_19_landmarks.csv'
with coordinate_export.open('w',newline='',encoding='utf-8') as handle:
    writer=csv.DictWriter(handle,fieldnames=list(coordinate_rows[0])); writer.writeheader(); writer.writerows(coordinate_rows)
print('Saved all 19 source-coordinate landmarks:',coordinate_export)

Inference complete: ID 1, source 1067×1175
Inference complete: ID 2, source 549×582
Inference complete: ID 3, source 515×576
Inference complete: ID 4, source 553×622
Inference complete: ID 6, source 516×593
Inference complete: ID 7, source 561×621
Inference complete: ID 8, source 687×768
Inference complete: ID 10, source 666×754
Inference complete: ID 11, source 549×601
Inference complete: ID 12, source 495×568
Inference complete: ID 13, source 544×594
Inference complete: ID 14, source 543×605
Inference complete: ID 17, source 507×569
Inference complete: ID 18, source 511×601
Inference complete: ID 19, source 660×766
Inference complete: ID 20, source 602×666
Inference complete: ID 21, source 563×620
Inference complete: ID 23, source 552×596
Inference complete: ID 26, source 551×609
Inference complete: ID 27, source 693×757
Inference complete: ID 28, source 521×602
Inference complete: ID 29, source 559×608
Inference complete: ID 30, source 567×616
Inference complete: ID 32, source 559×6

## 3. Paired SNA/SNB/signed ANB agreement and categories
Only the pre-specified Sella, Nasion, A-point, and B-point are used for angles. Categories use unrounded signed ANB and the approved 0°/4° limits.

In [10]:
patients,input_audit=ca.validate_coordinate_table(coordinate_rows,expected_ids=COHORT_IDS,require_source_geometry=True,coordinate_system='source_image_pixels_xy')
patient_rows=ca.compute_patient_angles(patients)
if len(patient_rows)!=60: raise ValueError(f'Incomplete validated cohort: {len(patient_rows)} of 60')
angular=ca.agreement_analysis(patient_rows,seed=SEED,bootstrap_repetitions=BOOTSTRAP_REPETITIONS)
category=ca.category_analysis(patient_rows,seed=SEED,bootstrap_repetitions=BOOTSTRAP_REPETITIONS)
discordant=ca.discordant_cases(patient_rows)
def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    fields=list(rows[0]) if rows else list(fieldnames or [])
    with Path(path).open('w',newline='',encoding='utf-8') as handle:
        writer=csv.DictWriter(handle,fieldnames=fields); writer.writeheader(); writer.writerows(rows)
audit_fields=['row_number','image_id','landmark_name','status','reason']
write_csv(OUTPUT_DIR/'coordinate_input_audit.csv',input_audit,audit_fields)
patient_fields=['image_id','reference_sna_deg','reference_snb_deg','reference_anb_deg','predicted_sna_deg','predicted_snb_deg','predicted_anb_deg','sna_signed_error_deg','snb_signed_error_deg','anb_signed_error_deg','reference_category','predicted_category','metadata']
write_csv(OUTPUT_DIR/'patient_angles.csv',patient_rows,patient_fields)
write_csv(OUTPUT_DIR/'angular_agreement.csv',[{'measure':name,**value['summary'],'bootstrap_95_ci':json.dumps(ca.json_safe(value['bootstrap_95_ci']),sort_keys=True)} for name,value in angular.items()])
write_csv(OUTPUT_DIR/'category_agreement.csv',[{'labels':json.dumps(category['labels']),'confusion_counts':json.dumps(category['confusion_counts']),'reference_row_rates':json.dumps(category['reference_row_rates']),'n':category['n'],'overall_agreement':category['overall_agreement'],'kappa_unweighted':category['kappa_unweighted'],'per_class':json.dumps(ca.json_safe(category['per_class'])),'bootstrap_95_ci':json.dumps(ca.json_safe(category['bootstrap_95_ci']),sort_keys=True)}])
discordant_schema=['image_id','reference_sna_deg','reference_snb_deg','reference_anb_deg','predicted_sna_deg','predicted_snb_deg','predicted_anb_deg','sna_signed_error_deg','snb_signed_error_deg','anb_signed_error_deg','reference_category','predicted_category','reference_distance_to_0_deg','predicted_distance_to_0_deg','reference_distance_to_4_deg','predicted_distance_to_4_deg']
write_csv(OUTPUT_DIR/'discordant_cases.csv',discordant,discordant_schema)
cp.save_angular_figures(patient_rows,OUTPUT_DIR)
cp.save_anb_category_figure(patient_rows,OUTPUT_DIR)
print('N paired:',len(patient_rows),'| discordant categories:',len(discordant))
print(json.dumps(ca.json_safe(category),indent=2))

N paired: 60 | discordant categories: 17
{
  "labels": [
    "Class III",
    "Class I",
    "Class II"
  ],
  "confusion_counts": [
    [
      5,
      6,
      0
    ],
    [
      1,
      23,
      5
    ],
    [
      0,
      5,
      15
    ]
  ],
  "reference_row_rates": [
    [
      0.45454545454545453,
      0.5454545454545454,
      0.0
    ],
    [
      0.034482758620689655,
      0.7931034482758621,
      0.1724137931034483
    ],
    [
      0.0,
      0.25,
      0.75
    ]
  ],
  "n": 60,
  "overall_agreement": 0.7166666666666667,
  "kappa_unweighted": 0.5251396648044693,
  "per_class": {
    "Class III": {
      "support": 11,
      "recall": 0.45454545454545453,
      "precision": 0.8333333333333334,
      "f1": 0.5882352941176471
    },
    "Class I": {
      "support": 29,
      "recall": 0.7931034482758621,
      "precision": 0.6764705882352942,
      "f1": 0.7301587301587301
    },
    "Class II": {
      "support": 20,
      "recall": 0.75,
      "precision": 

## 4. Provenance and factual summary
The manifest records the selected best-MRE checkpoint, exact experiment split, cohort role, geometry, and software. This analysis reports annotation-referenced validation agreement only; no diagnosis, equivalence, or clinical tolerance claim is made.

In [11]:
def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
manifest={'checkpoint_path':str(CHECKPOINT),'checkpoint_sha256':sha256(CHECKPOINT),'experiment':'finetune_n20','training_ids':list(TRAIN_IDS),'validation_ids':list(COHORT_IDS),'cohort_size':len(COHORT_IDS),'cohort_role':'validation cohort used for model development/model selection; not independent test data','coordinate_system':'source_image_pixels_xy','spacing_x_y':list(SPACING),'spacing_units':SPACING_UNITS,'physical_spacing_x_y':PHYSICAL_SPACING,'spacing_unit_name':PHYSICAL_SPACING_UNITS if SPACING_UNITS=='physical' else None,'coordinate_origin':COORDINATE_ORIGIN,'coordinate_orientation':COORDINATE_ORIENTATION,'landmark_indices_zero_based':dict(CLINICAL_LANDMARK_INDICES),'anb_categories_unrounded':{'Class III':'ANB < 0°','Class I':'0° <= ANB <= 4°','Class II':'ANB > 4°'},'seed':SEED,'bootstrap_repetitions':BOOTSTRAP_REPETITIONS,'python':platform.python_version(),'numpy':np.__version__,'training_or_checkpoint_selection_performed':False}
summary={'cohort_n':len(patient_rows),'cohort_role':manifest['cohort_role'],'definitions':{'SNA':'interior angle at Nasion, Nasion-Sella to Nasion-Subspinale','SNB':'interior angle at Nasion, Nasion-Sella to Nasion-Supramentale','ANB':'signed SNA - SNB','category_thresholds_unrounded':manifest['anb_categories_unrounded']},'angular_agreement':angular,'category_agreement':category,'discordant_n':len(discordant),'caveats':['Reference-derived category agreement is not complete diagnostic accuracy or Angle occlusal classification.','The 60-image cohort is development/model-selection validation, not independent testing.','Reference annotations are not error-free anatomical truth.','Bland-Altman limits are descriptive, not clinical acceptability thresholds.']}
ca.write_json(OUTPUT_DIR/'analysis_summary.json',summary)
ca.write_json(OUTPUT_DIR/'run_manifest.json',manifest)
report=['# Clinical angular analysis summary','',f'- Paired validation images: {len(patient_rows)}.',f'- Category-discordant images: {len(discordant)}.','- The cohort was used during model development/model selection and is not independent testing.','- Reference annotations are comparison references, not error-free anatomical truth.','- Categories use unrounded signed ANB and are not full clinical diagnoses or Angle classifications.','- Bland–Altman limits are descriptive, not clinical acceptability thresholds.','','## Angular agreement']
for name,result in angular.items(): report.append(f'- {name}: {json.dumps(ca.json_safe(result),sort_keys=True)}')
report += ['', '## ANB-derived category agreement', json.dumps(ca.json_safe(category),sort_keys=True), '', 'Interpret class metrics with support; rare-class reliability is not implied.', '', 'See analysis_summary.json for structured metrics and run_manifest.json for provenance.']
(OUTPUT_DIR/'results_summary.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
print('Results folder:',OUTPUT_DIR)
print((OUTPUT_DIR/'results_summary.md').read_text())

Results folder: /kaggle/working/clinical_angular_analysis/validation_best_mre_checkpoint
# Clinical angular analysis summary

- Paired validation images: 60.
- Category-discordant images: 17.
- The cohort was used during model development/model selection and is not independent testing.
- Reference annotations are comparison references, not error-free anatomical truth.
- Categories use unrounded signed ANB and are not full clinical diagnoses or Angle classifications.
- Bland–Altman limits are descriptive, not clinical acceptability thresholds.

## Angular agreement
- SNA: {"bootstrap_95_ci": {"bias_deg": {"lower": 0.9284737386606327, "undefined_replicates": 0, "upper": 2.5215906895066276, "valid_replicates": 5000}, "mae_deg": {"lower": 2.3830193899525476, "undefined_replicates": 0, "upper": 3.431553002531882, "valid_replicates": 5000}, "rmse_deg": {"lower": 3.0183049005974816, "undefined_replicates": 0, "upper": 4.114073488757518, "valid_replicates": 5000}}, "summary": {"bias_deg": 1.74

# Approved Kaggle radiograph overlay appendix

This independent, run-only appendix visualizes the approved saved angular results for illustrative cases 37, 4, 11, 17, and 3. It reads only `patient_angles.csv` and `source_coordinate_predictions_all_19_landmarks.csv`; it does not load a model, perform inference, recompute reported angle values, or modify any existing notebook cell. The audited [`clinical_analysis.py`](../clinical_angular_analysis/src/clinical_analysis.py) functions provide angle geometry for drawing the true smaller-angle arcs.

The figure pairs are visual comparisons, not diagnostic outputs. Reference annotations are comparison references, not error-free anatomical truth. This cohort informed model development/model selection and is not independent testing; selected cases are illustrative and not a representative performance sample. Relative spacing `(1, 1)` expresses pixel geometry only; acquisition physical square-pixel geometry is **UNVERIFIED**. No mm calibration or clinical diagnosis is implied.


In [ ]:
# Independent setup: paths, optional plotting dependencies, audited clinical helpers, and saved CSV inputs.
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import math
import os
import sys
import warnings

DEFAULT_IMAGES_ROOT = Path("/kaggle/input/datasets/pouyansajadi78/fine-tuning-dataset-final/images_v2/images_v2")
DEFAULT_RESULTS_DIR = Path("/kaggle/working/clinical_angular_analysis/validation_best_mre_checkpoint")
IMAGES_ROOT = Path(os.environ.get("CLINICAL_OVERLAY_IMAGES_ROOT", str(DEFAULT_IMAGES_ROOT))).expanduser()
RESULTS_DIR = Path(os.environ.get("CLINICAL_OVERLAY_RESULTS_DIR", str(DEFAULT_RESULTS_DIR))).expanduser()
DEFAULT_ANALYSIS_SOURCE_ROOT = Path("/kaggle/input/models/pouyansajadi78/model-oct-22-v2/tensorflow2/default/1/src")
ANALYSIS_SOURCE_ROOT = Path(os.environ.get("CLINICAL_ANALYSIS_SOURCE_ROOT", str(DEFAULT_ANALYSIS_SOURCE_ROOT))).expanduser()

try:
    from PIL import Image
    import matplotlib.pyplot as plt
    from matplotlib.patches import Arc
except ImportError as exc:
    raise ImportError(
        "Radiograph overlays require Pillow and Matplotlib in this Kaggle kernel. "
        "Attach an environment containing both dependencies; no legacy TIFF fallback is used."
    ) from exc

analysis_path = ANALYSIS_SOURCE_ROOT / "clinical_analysis.py"
if not analysis_path.is_file():
    raise FileNotFoundError(
        f"Audited clinical_analysis.py not found at {analysis_path}. Attach the approved model/source "
        "dataset or set CLINICAL_ANALYSIS_SOURCE_ROOT to its src directory; do not substitute an "
        "independent angle implementation."
    )
spec = importlib.util.spec_from_file_location("audited_clinical_analysis", analysis_path)
clinical_analysis = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = clinical_analysis
spec.loader.exec_module(clinical_analysis)
interior_angle = clinical_analysis.interior_angle
angles_from_landmarks = clinical_analysis.angles_from_landmarks
classify_anb = clinical_analysis.classify_anb

PATIENT_ANGLES_CSV = RESULTS_DIR / "patient_angles.csv"
COORDINATES_CSV = RESULTS_DIR / "source_coordinate_predictions_all_19_landmarks.csv"
for required_csv in (PATIENT_ANGLES_CSV, COORDINATES_CSV):
    if not required_csv.is_file():
        raise FileNotFoundError(
            f"Required approved saved CSV is missing: {required_csv}. Mount the CSVs under the "
            "configured results folder or set CLINICAL_OVERLAY_RESULTS_DIR."
        )
with PATIENT_ANGLES_CSV.open(newline="", encoding="utf-8-sig") as handle:
    patient_rows = list(csv.DictReader(handle))
with COORDINATES_CSV.open(newline="", encoding="utf-8-sig") as handle:
    coordinate_rows = list(csv.DictReader(handle))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def index_unique_rows(rows, label):
    indexed = {}
    for row in rows:
        image_id = str(row.get("image_id", "")).strip()
        if not image_id:
            raise ValueError(f"{label} contains a blank image_id")
        if image_id in indexed:
            raise ValueError(f"{label} contains duplicate image_id {image_id!r}")
        indexed[image_id] = row
    return indexed

angles_by_id = index_unique_rows(patient_rows, "patient_angles.csv")
selected_ids = ["37", "4", "11", "17", "3"]
missing_ids = [image_id for image_id in selected_ids if image_id not in angles_by_id]
if missing_ids:
    raise ValueError(f"Selected image IDs absent from patient_angles.csv: {missing_ids}")
if not IMAGES_ROOT.is_dir():
    raise FileNotFoundError(f"Configured final images root does not exist: {IMAGES_ROOT}")

warnings.warn(
    "Acquisition physical square-pixel geometry is UNVERIFIED. Relative (1, 1) spacing "
    "supports source-pixel angular visualization only; it does not establish mm calibration.",
    RuntimeWarning,
)


In [ ]:
# Validate coordinate rows, uniquely resolve source images, and prepare the shared geometry for each panel pair.
from collections import defaultdict

coordinates_by_id = defaultdict(list)
for row in coordinate_rows:
    image_id = str(row.get("image_id", "")).strip()
    if not image_id:
        raise ValueError("source_coordinate_predictions_all_19_landmarks.csv contains blank image_id")
    coordinates_by_id[image_id].append(row)

EXPECTED_METADATA = {
    "coordinate_system": "source_image_pixels_xy",
    "origin": "top_left_pixel_center",
    "orientation": "x_right_y_down",
    "spacing_units": "dimensionless_relative",
}

def validate_case_rows(image_id):
    rows = coordinates_by_id.get(image_id, [])
    if len(rows) != 19:
        raise ValueError(f"Image {image_id}: expected exactly 19 landmark rows, found {len(rows)}")
    names = [row.get("landmark_name", "").strip() for row in rows]
    if len(set(names)) != 19 or set(names) != set(clinical_analysis.CANONICAL_LANDMARKS):
        raise ValueError(f"Image {image_id}: landmark identities must be the unique canonical 19")
    first = rows[0]
    for key, expected in EXPECTED_METADATA.items():
        if first.get(key) != expected:
            raise ValueError(f"Image {image_id}: expected {key}={expected!r}, found {first.get(key)!r}")
    try:
        width, height = int(first["source_width"]), int(first["source_height"])
        spacing = (float(first["x_spacing"]), float(first["y_spacing"]))
    except (KeyError, TypeError, ValueError) as exc:
        raise ValueError(f"Image {image_id}: invalid source dimensions or spacing metadata") from exc
    if width <= 0 or height <= 0 or spacing != (1.0, 1.0):
        raise ValueError(f"Image {image_id}: dimensions must be positive and relative spacing exactly (1, 1)")
    for row in rows:
        for key, expected in EXPECTED_METADATA.items():
            if row.get(key) != expected:
                raise ValueError(f"Image {image_id}: inconsistent {key} metadata")
        if int(row["source_width"]) != width or int(row["source_height"]) != height:
            raise ValueError(f"Image {image_id}: inconsistent source dimensions across coordinate rows")
        if (float(row["x_spacing"]), float(row["y_spacing"])) != spacing:
            raise ValueError(f"Image {image_id}: inconsistent axis spacing across coordinate rows")
    return rows, width, height

def resolve_image(image_id):
    variants = (f"({image_id})", image_id)
    extensions = ("tif", "tiff", "png", "jpg", "jpeg")
    candidates = [IMAGES_ROOT / f"{stem}.{extension}" for stem in variants for extension in extensions]
    matches = [path.resolve() for path in candidates if path.is_file()]
    if not matches:
        raise FileNotFoundError(f"Image {image_id}: no supported image found in {IMAGES_ROOT}")
    if len(matches) != 1:
        raise RuntimeError(f"Image {image_id}: ambiguous image variants found: {[str(p) for p in matches]}")
    return matches[0]

def xy(row, prefix):
    point = (float(row[f"{prefix}_x"]), float(row[f"{prefix}_y"]))
    if not all(math.isfinite(value) for value in point):
        raise ValueError(f"Non-finite {prefix} coordinate for image {row.get('image_id')}")
    return point

def points_for(rows, kind):
    prefix = "reference" if kind == "reference" else "predicted"
    return {
        "Sella": xy(next(r for r in rows if r["landmark_name"] == "Sella"), prefix),
        "Nasion": xy(next(r for r in rows if r["landmark_name"] == "Nasion"), prefix),
        "Subspinale": xy(next(r for r in rows if r["landmark_name"] == "Subspinale"), prefix),
        "Supramentale": xy(next(r for r in rows if r["landmark_name"] == "Supramentale"), prefix),
    }

def format_signed(value):
    value = float(value)
    return f"{value:+.2f}°"

prepared_cases = {}
for image_id in selected_ids:
    rows, csv_width, csv_height = validate_case_rows(image_id)
    image_path = resolve_image(image_id)
    with Image.open(image_path) as image_handle:
        image_size = image_handle.size
        image = image_handle.convert("L").copy()
    if image_size != (csv_width, csv_height):
        raise ValueError(
            f"Image {image_id}: image dimensions {image_size} do not exactly match CSV "
            f"(width, height)=({csv_width}, {csv_height})"
        )
    ref_points = points_for(rows, "reference")
    pred_points = points_for(rows, "predicted")
    # Exercise audited angle helpers for geometry validation only; displayed values come solely from saved patient_angles.csv.
    for points in (ref_points, pred_points):
        angles_from_landmarks(points, spacing=(1.0, 1.0))
    prepared_cases[image_id] = {
        "image": image,
        "image_path": image_path,
        "image_sha256": sha256_file(image_path),
        "width": csv_width,
        "height": csv_height,
        "rows": rows,
        "reference": ref_points,
        "prediction": pred_points,
        "values": angles_by_id[image_id],
    }


In [ ]:
# Render and save non-overwriting PNG/PDF figures with paired shared crops and explicit provenance.
import numpy as np


def padded_union_limits(case, pad_fraction=0.12, minimum_padding=18.0):
    all_points = [*case["reference"].values(), *case["prediction"].values()]
    xs = [point[0] for point in all_points]
    ys = [point[1] for point in all_points]
    span_x = max(max(xs) - min(xs), 1.0)
    span_y = max(max(ys) - min(ys), 1.0)
    pad_x = max(minimum_padding, span_x * pad_fraction)
    pad_y = max(minimum_padding, span_y * pad_fraction)
    return (min(xs) - pad_x, max(xs) + pad_x, min(ys) - pad_y, max(ys) + pad_y)

def add_angle_arc(ax, points, first_name, second_name, radius):
    vertex = points["Nasion"]
    first = points[first_name]
    second = points[second_name]
    # Determine the smaller signed sweep from actual ray vectors in display coordinates (y-down image axes).
    v1 = np.asarray(first, dtype=float) - np.asarray(vertex, dtype=float)
    v2 = np.asarray(second, dtype=float) - np.asarray(vertex, dtype=float)
    a1, a2 = math.degrees(math.atan2(v1[1], v1[0])), math.degrees(math.atan2(v2[1], v2[0]))
    sweep = (a2 - a1 + 180.0) % 360.0 - 180.0
    audited = interior_angle(vertex, first, second, spacing=(1.0, 1.0))
    if not math.isclose(abs(sweep), audited, rel_tol=1e-7, abs_tol=1e-7):
        raise ValueError("Arc sweep disagrees with audited smaller interior angle")
    arc = Arc(vertex, 2 * radius, 2 * radius, angle=0, theta1=a1, theta2=a1 + sweep,
              color="#ffdd00", linewidth=2.0, zorder=8)
    ax.add_patch(arc)
    mid = math.radians(a1 + sweep / 2)
    ax.text(vertex[0] + radius * 1.18 * math.cos(mid),
            vertex[1] + radius * 1.18 * math.sin(mid), f"{audited:.1f}°",
            color="white", fontsize=8, ha="center", va="center",
            bbox={"boxstyle": "round,pad=0.15", "facecolor": "#333333", "edgecolor": "none", "alpha": 0.88}, zorder=10)

def draw_panel(ax, case, points, title, predicted):
    image = case["image"]
    ax.imshow(image, cmap="gray", origin="upper", vmin=0, vmax=255, interpolation="nearest")
    s, n, a, b = points["Sella"], points["Nasion"], points["Subspinale"], points["Supramentale"]
    color = "#666666" if predicted else "#171717"
    linestyle = "--" if predicted else "-"
    marker = "s" if predicted else "o"
    linewidth = 1.8
    for endpoint in (a, b):
        ax.plot([n[0], s[0]], [n[1], s[1]], color=color, linestyle=linestyle, linewidth=linewidth, zorder=4)
        ax.plot([n[0], endpoint[0]], [n[1], endpoint[1]], color=color, linestyle=linestyle, linewidth=linewidth, zorder=4)
    for point, label in ((s, "S"), (n, "N"), (a, "A"), (b, "B")):
        ax.scatter(point[0], point[1], s=42, marker=marker, facecolor=("white" if predicted else "#171717"),
                   edgecolor=color, linewidth=1.4, zorder=9)
        ax.annotate(label, point, xytext=(5, -7), textcoords="offset points", color="white", fontsize=10,
                    fontweight="bold", bbox={"boxstyle": "round,pad=0.1", "facecolor": "#222222", "edgecolor": "none", "alpha": 0.78}, zorder=11)
    radius = max(9.0, min(case["width"], case["height"]) * 0.045)
    add_angle_arc(ax, points, "Sella", "Subspinale", radius)
    add_angle_arc(ax, points, "Sella", "Supramentale", radius * 1.28)
    # Intentional: no ANB arc is drawn.
    xmin, xmax, ymin, ymax = case["limits"]
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymax, ymin)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Source x (pixels; rightward)")
    ax.set_ylabel("Source y (pixels; downward)")
    ax.grid(False)


def value_box(values):
    ref_anb = float(values["reference_anb_deg"])
    pred_anb = float(values["predicted_anb_deg"])
    ref_cat = values["reference_category"]
    pred_cat = values["predicted_category"]
    return (
        f"Reference  SNA {float(values['reference_sna_deg']):.2f}°  |  "
        f"SNB {float(values['reference_snb_deg']):.2f}°  |  ANB {format_signed(ref_anb)}  |  {ref_cat}\n"
        f"Prediction SNA {float(values['predicted_sna_deg']):.2f}°  |  "
        f"SNB {float(values['predicted_snb_deg']):.2f}°  |  ANB {format_signed(pred_anb)}  |  {pred_cat}\n"
        f"Saved signed error: ANB {format_signed(values['anb_signed_error_deg'])}  |  "
        f"Category transition: {ref_cat} → {pred_cat}"
    )

output_root = (RESULTS_DIR / "figures_clinician" / "radiograph_overlays").resolve()
results_root = RESULTS_DIR.resolve()
if not output_root.is_relative_to(results_root):
    raise RuntimeError(f"Overlay output escaped configured results folder: {output_root}")
output_root.mkdir(parents=True, exist_ok=True)
manifest = {
    "appendix": "approved_kaggle_radiograph_overlay",
    "selected_ids": selected_ids,
    "selection_caveat": "Illustrative selected cases only; not representative cohort performance or independent testing.",
    "geometry": {
        "coordinate_system": "source_image_pixels_xy",
        "origin": "top_left_pixel_center",
        "orientation": "x_right_y_down",
        "relative_spacing": [1.0, 1.0],
        "equal_axis_aspect": True,
        "shared_pair_crop": "padded union of reference and prediction S/N/A/B points",
        "acquisition_physical_square_pixel_geometry": "UNVERIFIED",
        "caveat": "Relative spacing is dimensionless and does not provide mm calibration.",
    },
    "csv_inputs": {
        "patient_angles.csv": {"path": str(PATIENT_ANGLES_CSV.resolve()), "sha256": sha256_file(PATIENT_ANGLES_CSV)},
        "source_coordinate_predictions_all_19_landmarks.csv": {"path": str(COORDINATES_CSV.resolve()), "sha256": sha256_file(COORDINATES_CSV)},
    },
    "audited_analysis_source": {"path": str(analysis_path.resolve()), "sha256": sha256_file(analysis_path)},
    "cases": {},
}
figure_objects = []
for image_id in selected_ids:
    case = prepared_cases[image_id]
    case["limits"] = padded_union_limits(case)
    values = case["values"]
    # Confirm saved signed error and category fields are internally aligned without replacing their saved display values.
    if not math.isclose(float(values["anb_signed_error_deg"]),
                        float(values["predicted_anb_deg"]) - float(values["reference_anb_deg"]), abs_tol=1e-8):
        raise ValueError(f"Image {image_id}: saved ANB signed error is inconsistent with saved angle values")
    if classify_anb(float(values["reference_anb_deg"])) != values["reference_category"] or classify_anb(float(values["predicted_anb_deg"])) != values["predicted_category"]:
        raise ValueError(f"Image {image_id}: saved category does not match audited unrounded ANB classification")
    base = f"case_{image_id}_reference_vs_prediction"
    png_path = (output_root / f"{base}.png").resolve()
    pdf_path = (output_root / f"{base}.pdf").resolve()
    for destination in (png_path, pdf_path):
        if not destination.is_relative_to(results_root):
            raise RuntimeError(f"Output path escaped configured results folder: {destination}")
        if destination.exists():
            raise FileExistsError(f"Refusing to overwrite existing overlay output: {destination}")
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5), constrained_layout=True)
    draw_panel(axes[0], case, case["reference"], f"Case {image_id} — Reference", predicted=False)
    draw_panel(axes[1], case, case["prediction"], f"Case {image_id} — Prediction", predicted=True)
    fig.suptitle("Source radiograph landmark overlay — illustrative comparison", fontsize=13, fontweight="bold")
    fig.text(0.5, 0.015, value_box(values), ha="center", va="bottom", fontsize=9,
             bbox={"boxstyle": "round,pad=0.45", "facecolor": "#f2f2f2", "edgecolor": "#555555"})
    fig.savefig(png_path, dpi=300, format="png")
    try:
        fig.savefig(pdf_path, format="pdf")
    except Exception:
        png_path.unlink(missing_ok=True)
        raise
    figure_objects.append(fig)
    manifest["cases"][image_id] = {
        "image_absolute_path": str(case["image_path"]),
        "image_sha256": case["image_sha256"],
        "dimensions_width_height": [case["width"], case["height"]],
        "saved_values": {key: values[key] for key in (
            "reference_sna_deg", "reference_snb_deg", "reference_anb_deg",
            "predicted_sna_deg", "predicted_snb_deg", "predicted_anb_deg",
            "anb_signed_error_deg", "reference_category", "predicted_category")},
        "outputs": {"png": str(png_path), "pdf": str(pdf_path)},
        "shared_axis_extent_xy": list(case["limits"]),
    }

manifest_path = (output_root / "manifest.json").resolve()
if not manifest_path.is_relative_to(results_root):
    raise RuntimeError(f"Manifest output escaped configured results folder: {manifest_path}")
if manifest_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing overlay manifest: {manifest_path}")
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Saved overlay figures and provenance manifest under: {output_root}")
plt.show()
